# Imports and Reading data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
plt.style.use('ggplot')
pd.set_option('display.max_columns', 200)


In [ ]:
df_gain1amp02fre6 = pd.read_csv('OTO-ESC_fix-1_0.2_6-100kshots.csv')


# Data understanding

In [ ]:
df_gain1amp02fre6.shape


In [ ]:
df_gain1amp02fre6.columns


In [ ]:
df_gain1amp02fre6.head(5)


# Feature understanding

In [ ]:
dfs = {
    'gain 1 amp 0.2 fre 6': df_gain1amp02fre6,
}


In [ ]:
cv = df_gain1amp02fre6['choose_ESC'].value_counts().head(10).plot(kind='bar', title='Convergence count')
cv.set_xlabel('ESC outperforms the fixed fuel-ratio benchmark')
cv.set_ylabel('Counts')


# Feature relationship

## Energy vs. other features

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)

for ax, (name, df) in zip(axes.flat, dfs.items()):
    df.plot(
        kind='scatter',
        x='convergence_time',
        y='energy_ESC',
        ax=ax,
        title=name
    )
    ax.set_xlabel('Convergence time')
    ax.set_ylabel('Energy generated using ESC')

plt.tight_layout()
plt.show()


In [ ]:
params = ['optimizer_gain', 'dither_amplitude', 'dither_frequency']

for p in params:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)

    for ax, (name, df) in zip(axes.flat, dfs.items()):
        df.plot(
            kind='scatter',
            x=p,
            y='energy_ESC',
            ax=ax,
            title=name
        )
        ax.set_xlabel(p)
        ax.set_ylabel('Energy generated using ESC')

    fig.suptitle(f'Energy generated using ESC vs {p}')
    plt.tight_layout()
    plt.show()


## Symmetrical ESC parameter analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# ----- Setup -----

feature_cols = ['optimizer_gain', 'dither_amplitude', 'dither_frequency']
target = 'choose_ESC'

pairs = [
    ('gain 1 amp 0.2 fre 6', 'gain 1 amp 0.2 fre 6'),
]

def to_tensors(df):
    X = torch.tensor(df[feature_cols].values, dtype=torch.float32)
    y = torch.tensor(df[target].values, dtype=torch.float32).reshape(-1, 1)
    return X, y

def make_model(n):
    return nn.Sequential(
        nn.Linear(n, 32), nn.ReLU(),
        nn.Linear(32, 16), nn.ReLU(),
        nn.Linear(16, 1)
    )

def train(model, train_loader, Xtr, ytr, Xcv, ycv, epochs=300, lr=0.01):
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {'loss': [], 'val_loss': []}

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            outputs = model(xb)
            loss = loss_fn(outputs, yb)
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            model.eval()
            train_loss = loss_fn(model(Xtr), ytr)
            val_loss = loss_fn(model(Xcv), ycv)
        history['loss'].append(float(train_loss))
        history['val_loss'].append(float(val_loss))
    return history


In [ ]:
# ----- Train on one dataset of each pair, test on the other -----

results = {}

for train_name, test_name in pairs:
    df_train = dfs[train_name][feature_cols + [target]]
    df_test = dfs[test_name][feature_cols + [target]]

    tr, cv = train_test_split(df_train, test_size=0.20, random_state=1)

    Xtr_p, ytr_p = to_tensors(tr)
    Xcv_p, ycv_p = to_tensors(cv)
    Xte_p, _ = to_tensors(df_test)

    loader = DataLoader(TensorDataset(Xtr_p, ytr_p), batch_size=100, shuffle=True)

    torch.manual_seed(1)
    model = make_model(Xtr_p.shape[1])

    history = train(model, loader, Xtr_p, ytr_p, Xcv_p, ycv_p)

    plt.plot(history['loss'], label='train loss')
    plt.plot(history['val_loss'], label='cv loss')
    plt.title(f'Trained on {train_name}')
    plt.legend()
    plt.show()

    model.eval()
    with torch.no_grad():
        pred = (torch.sigmoid(model(Xte_p)) >= 0.5).float().numpy().ravel()
    acc = (pred == df_test[target].to_numpy()).mean()
    results[(train_name, test_name)] = acc
    print(f'{train_name} -> {test_name}: Accuracy = {acc:.4f}')


# Visualization

In [ ]:
import plotly.express as px

for name, d in dfs.items():
    d_plot = d.assign(choose_ESC=d['choose_ESC'].astype(int).astype(str))

    fig = px.scatter_3d(
        d_plot,
        x='optimizer_gain',
        y='dither_amplitude',
        z='dither_frequency',
        color='choose_ESC',
        color_discrete_map={'0': '#3b4252', '1': '#3b7dd8'},
        template='plotly_dark',
        category_orders={'choose_ESC': ['0', '1']},
        title=f'3D scatter: choose ESC ({name})',
        labels={
            'optimizer_gain': 'Optimizer gain',
            'dither_amplitude': 'Dither amplitude',
            'dither_frequency': 'Dither Frequency',
            'choose_ESC': 'Choose ESC'
        }
    )
    fig.update_traces(marker=dict(size=3))
    fig.show()
